# 2026年9月15日 2026機械学習PJ 第3回 画像処理，自然言語処理 説明ノートブック

本ノートブックでは，以下の内容を知る/触れることを目的とする:

1. 画像処理では，画像がどのような数値データとして扱われるかを知る．
2. CNNによる画像分類を実際に学習し，予測結果や誤分類を確認する．
3. Vision Transformerでは，画像をパッチへ分割してTransformerで扱うという考え方を知る．
4. 自然言語処理では，文章をトークンへ分割し，Transformerで扱う基本的な考え方を知る．
5. 画像と言語の両方を扱うモデルとしてCLIPを動かし，画像と文章を同じ空間で比較するイメージをつかむ．
6. 画像と文章の場合に限らず，マルチモーダルモデルとして組み合わせて扱う場合でも，モデル構築の基本的な流れは共通していることを理解する．

取り組む皆さんに注意してほしいこと:

- 今回も，数式の厳密な導出より，実際にモデルを動かして処理のイメージをつかむことを重視します．
- CNNの畳み込み演算やTransformerのAttention機構について，数理的な詳細までは扱いません．
- コードを一字一句暗記する必要はありません．何を入力し，どのモデルで処理し，何が出力されるのかを意識してください．
- Google ColabのT4 GPUで，1時間半程度の時間内に十分な余裕を持って実行できる内容を想定しています．
- 言語モデルとCLIPは初回実行時に事前学習済みモデルをダウンロードするため，通信環境によって少し時間がかかります．

作成者: 坪井 一馬 (本資料は生成AIを使用して作成した部分がありますが，最終的な責任は坪井に帰属します)

## 0. 今回の全体像

今回は内容を3つに分けます．いずれの場合も，**入力された情報を数値に変換する**ということが必要になります．

1. **画像処理**  
   Fashion-MNISTを使って，畳み込みニューラルネットワーク(CNN)による画像分類を実際に学習します．さらに，Vision Transformerと呼ばれるモデルが画像をどのように扱うかを直感的に確認します．

2. **自然言語処理**  
   文章を数値で表現可能なトークンへ分割し，事前学習済みTransformerを使って感情分類を実行します．

3. **画像と言語の両方を扱うモデル**  
   CLIPを使って，1枚の画像と複数の文章の対応度を比較してみます．

第1回，第2回と同様に，どの分野でも次の4段階を意識します．

1. データを準備する．  
2. モデルを準備する．  
3. (必要であれば)モデルを学習する．  
4. テストや推論を行う．

## 使用するライブラリの読み込み

必要となるPythonの機能を使えるようにする．

⭐️**重要**⭐️ **最初にGoogle Colabの設定において「T4 GPU」を使用する設定にしてください．画面右上にある「▼」から選択できます．**

In [ ]:
# Python標準ライブラリ
import copy
import importlib.util
import random
import subprocess
import sys

# 数値計算・可視化
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# PyTorch
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import (
    DataLoader,
    Subset,
    random_split,
)
from torchvision import datasets, transforms

# Transformers
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, CLIPModel, CLIPProcessor

In [ ]:
# 実行ごとの差を小さくするため，乱数を固定する
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# T4 GPUが利用できる場合はGPU，利用できない場合はCPUを使用する
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("使用するデバイス:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. 画像処理

ここでは "Fashion-MNIST" と呼ばれる10クラスのデータセットを使って，畳み込みニューラルネットワーク(CNN)に基づく入力画像の10種類の分類モデルを作成し，実際に学習します．さらに，Vision Transformerと呼ばれるモデルが画像をどのように扱うかを直感的に確認します．

### 1.1. Fashion-MNISTを読み込む

まずはFashion-MNISTのデータを読み込み，実際の画像を確認します．

In [ ]:
# Fashion-MNISTでは，画像をPyTorchのTensorへ変換するだけにする
transform = transforms.ToTensor()

# 学習用として用意されている60,000枚を読み込む
full_train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

# テスト用として用意されている10,000枚を読み込む
test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

# 学習用60,000枚を，訓練50,000枚と評価10,000枚へ分割する
split_generator = torch.Generator().manual_seed(
    RANDOM_STATE
)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [50000, 10000],
    generator=split_generator,
)

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

print("訓練データ:", len(train_dataset))
print("評価データ:", len(val_dataset))
print("テストデータ:", len(test_dataset))

In [ ]:
# Fashion-MNISTの画像を12枚表示する
fig, axes = plt.subplots(
    3,
    4,
    figsize=(9, 7),
)

for index, ax in enumerate(axes.flat):
    image, label = train_dataset[index]

    ax.imshow(
        image.squeeze(),
        cmap="gray",
    )

    ax.set_title(
        class_names[label]
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

### 1.2. 画像が数値として表現されていることを確認する

人間は画像を目で見て「靴」「バッグ」などと判断できますが，モデル化するにあたっては，**入力された情報を数値に変換する**必要があります．画像の場合，モデルへ入力されるのは画素値になります．Fashion-MNISTにある各画像は，次の3つの次元を持つテンソルとして扱われます．

1. 色チャネル数：1
2. 高さ：28画素
3. 幅：28画素

ただし，Fashion-MNISTはすべてグレースケール画像なので，色チャネルは1つです．色のある画像の場合は，高さと幅，そして色チャネル数を調整することで扱うことができます．

ここで変換された数値を入力として，モデル内部で処理を行うことのできるようにします．

In [ ]:
image, label = train_dataset[0]

print("画像Tensorの形:", image.shape)
print("正解ラベル:", label)
print("正解クラス:", class_names[label])

plt.figure(figsize=(4, 4))
plt.imshow(
    image.squeeze(),
    cmap="gray",
)
plt.title(class_names[label])
plt.axis("off")
plt.show()

In [ ]:
# 画像の中央付近の5×5画素だけ，数値として表示する
print(
    image[
        0,
        10:15,
        10:15,
    ]
)

### 1.3. 畳み込みニューラルネットワーク(CNN)

畳み込みニューラルネットワーク(Convolutional Neural Network; CNN)は，画像の局所的な特徴を捉えるのが得意なニューラルネットワークであり，**入力された画像にフィルタを重ねて演算することを通して，特徴を取り出します**．そして，取り出された特徴に基づいて目的のタスクを学習します．

1. 畳み込み層 `Conv2d`  
  フィルタを入力画像に重ねることで，小さな範囲を見ながら特徴を取り出します．PyTorchではデフォルトで実装されているので，畳み込みの演算を自力で実装する必要はありません．
2. 活性化関数 `ReLU`  
  複雑な特徴を扱うことのできるように，非線形な変換を加えます．
3. プーリング層 `MaxPool2d`  
  重要な特徴を残しながら特徴マップを小さくします．
4. ここまでの処理の繰り返し
5. 平坦化 `Flatten`  
  特徴マップを1列の数値へ並べます．
6. 線形層 `Linear`  
  最終的な10クラスのスコアを出力します．最後の `Linear(64, 10)` の出力10個は，Fashion-MNISTの10クラスそれぞれに対応するlogitです．

ここで重要なのは，**重ねられるフィルタの値を人間が細かく決めるのではなく，分類が上手くいくように学習によって更新する**点です．タスクを精度よく解けるようにするため，フィルタがどうなっていればよりよい特徴が得られるかを学習しています．

In [ ]:
class FashionCNN(nn.Module):
    def __init__(
        self,
        base_filters=16,
    ):
        super().__init__()

        # 画像から特徴を取り出す部分
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=base_filters,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=base_filters,
                out_channels=base_filters * 2,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # 取り出した特徴から10クラスを分類する部分
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                base_filters * 2 * 7 * 7,
                64,
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                10,
            ),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("訓練データのミニバッチ数:", len(train_loader))

In [ ]:
BASE_FILTERS = 16
LEARNING_RATE = 0.001

model = FashionCNN(
    base_filters=BASE_FILTERS,
).to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print(model)

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "学習可能なパラメータ数:",
    total_parameters,
)

### 1.4. CNNを学習する

学習では次の処理を繰り返します．モデルの構造や「フィルタを重ねる」という点は個性的ではありますが，基本的な流れは，第2回で行ったような通常の全結合ニューラルネットワークの場合と変わりません．

1. `optimizer.zero_grad()`  
  前回までに保存されている勾配(損失関数を更新するためのもの)をリセットする．
2. `outputs = model(features)`  
  現在のモデルへ入力を与えて，予測値であるlogitを得る．
3. `loss = loss_function(outputs, labels)`  
  損失関数に基づいて，予測と正解のずれを損失として計算する．
4. `loss.backward()`  
  損失を小さくするために，各パラメータ(ここではフィルタ)をどちらへどの程度動かせばよいかを勾配として計算する．
5. `optimizer.step()`  
  計算された勾配を用いて，実際にモデルのパラメータ(ここではフィルタ)を更新する．

In [ ]:
def train_one_epoch(
    model,
    data_loader,
    loss_function,
    optimizer,
    device,
):
    # 学習モードへ切り替える
    model.train()

    total_loss = 0.0
    total_count = 0

    predictions_all = []
    labels_all = []

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. 前回の勾配をリセットする
        optimizer.zero_grad()

        # 2. 現在のモデルで予測する
        logits = model(images)

        # 3. 予測と正解のずれを計算する
        loss = loss_function(
            logits,
            labels,
        )

        # 4. 損失を小さくする修正方向を計算する
        loss.backward()

        # 5. 実際にパラメータを更新する
        optimizer.step()

        predictions = logits.argmax(
            dim=1
        )

        total_loss += (
            loss.item()
            * images.size(0)
        )

        total_count += labels.size(0)

        predictions_all.extend(
            predictions
            .detach()
            .cpu()
            .numpy()
        )

        labels_all.extend(
            labels
            .detach()
            .cpu()
            .numpy()
        )

    return {
        "loss": total_loss / total_count,
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
    }

In [ ]:
def evaluate_model(
    model,
    data_loader,
    loss_function,
    device,
):
    # 評価モードへ切り替える
    model.eval()

    total_loss = 0.0
    total_count = 0

    predictions_all = []
    labels_all = []

    # 評価時は勾配を計算しない
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            loss = loss_function(
                logits,
                labels,
            )

            predictions = logits.argmax(
                dim=1
            )

            total_loss += (
                loss.item()
                * images.size(0)
            )

            total_count += labels.size(0)

            predictions_all.extend(
                predictions
                .cpu()
                .numpy()
            )

            labels_all.extend(
                labels
                .cpu()
                .numpy()
            )

    return {
        "loss": total_loss / total_count,
        "accuracy": accuracy_score(
            labels_all,
            predictions_all,
        ),
        "predictions": np.array(
            predictions_all
        ),
        "labels": np.array(
            labels_all
        ),
    }

In [ ]:
MAX_EPOCHS = 4

train_loss_history = []
train_accuracy_history = []

val_loss_history = []
val_accuracy_history = []

best_val_loss = float("inf")
best_epoch = None
best_model_state = None

for epoch in range(MAX_EPOCHS):
    train_result = train_one_epoch(
        model,
        train_loader,
        loss_function,
        optimizer,
        device,
    )

    val_result = evaluate_model(
        model,
        val_loader,
        loss_function,
        device,
    )

    train_loss_history.append(
        train_result["loss"]
    )

    train_accuracy_history.append(
        train_result["accuracy"]
    )

    val_loss_history.append(
        val_result["loss"]
    )

    val_accuracy_history.append(
        val_result["accuracy"]
    )

    # 評価Lossが最も小さい時点のモデルを保存する
    if val_result["loss"] < best_val_loss:
        best_val_loss = val_result["loss"]
        best_epoch = epoch + 1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch + 1:2d} | "
        f"Train Loss {train_result['loss']:.4f} | "
        f"Val Loss {val_result['loss']:.4f} | "
        f"Train Acc {train_result['accuracy']:.3f} | "
        f"Val Acc {val_result['accuracy']:.3f}"
    )

print()
print("評価Lossが最小だったEpoch:", best_epoch)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_loss_history,
    marker="o",
    label="Train Loss",
)

plt.plot(
    val_loss_history,
    marker="o",
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss history")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_accuracy_history,
    marker="o",
    label="Train Accuracy",
)

plt.plot(
    val_accuracy_history,
    marker="o",
    label="Validation Accuracy",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy history")
plt.legend()
plt.show()

### 1.5. テスト画像を分類する

学習されたモデルの成果を確認するため，テストデータの画像を入力して実際に分類してみましょう．

In [ ]:
# 評価データで最も良かった時点のモデルへ戻す
model.load_state_dict(
    best_model_state
)

test_result = evaluate_model(
    model,
    test_loader,
    loss_function,
    device,
)

print(
    f"Test Loss: "
    f"{test_result['loss']:.4f}"
)

print(
    f"Test Accuracy: "
    f"{test_result['accuracy']:.3f}"
)

In [ ]:
# テストデータから16枚を取り出し，予測結果を表示する
model.eval()

fig, axes = plt.subplots(
    4,
    4,
    figsize=(10, 10),
)

with torch.no_grad():
    for index, ax in enumerate(axes.flat):
        image, true_label = test_dataset[index]

        logits = model(
            image.unsqueeze(0).to(device)
        )

        predicted_label = logits.argmax(
            dim=1
        ).item()

        ax.imshow(
            image.squeeze(),
            cmap="gray",
        )

        ax.set_title(
            f"True: {class_names[true_label]}\n"
            f"Pred: {class_names[predicted_label]}"
        )

        ax.axis("off")

plt.tight_layout()
plt.show()

### 1.6. 誤分類の事例を見る

Accuracyなどの統計的指標だけでは，どのような画像を間違えたか分かりません．**モデルの弱点を考えるためには，評価指標だけでなく具体的な出力を見ることも重要**です．モデルの性能を向上させたり，内部の作用機序を分析するための考察として，実際の誤分類事例を確認することは非常に重要です．

そこで，実際の誤分類画像を見ると，ShirtとT-shirt/top，PulloverとCoatなど，見た目が近いクラス同士で混同していることがあります．

In [ ]:
# テストデータ全体から誤分類された画像を探す
wrong_indices = np.where(
    test_result["predictions"]
    != test_result["labels"]
)[0]

print("誤分類数:", len(wrong_indices))

fig, axes = plt.subplots(
    4,
    4,
    figsize=(10, 10),
)

for position, ax in enumerate(axes.flat):
    dataset_index = int(
        wrong_indices[position]
    )

    image, true_label = test_dataset[
        dataset_index
    ]

    predicted_label = int(
        test_result["predictions"][
            dataset_index
        ]
    )

    ax.imshow(
        image.squeeze(),
        cmap="gray",
    )

    ax.set_title(
        f"True: {class_names[true_label]}\n"
        f"Pred: {class_names[predicted_label]}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

### 1.7. CNNの中間表現を見る

最初の畳み込み層が作った特徴マップを表示します．特徴マップは，**モデル内部において，入力された画像にフィルタを重ねて得られる画像の特徴を取り出したもの**です．

これはモデルの解釈内容を完全に説明するものではありませんが，入力画像が中間層で別の表現へ変換されていることを確認できます．元画像そのものではなく，明るさの変化や輪郭の一部などが強調されたような画像が複数得られます．

これらの表現を組み合わせたり，**前の層で得られた特徴マップを次の層の入力にして，新たな特徴マップを得る**などして，最終的に分類(目的のタスク)に役立つ潜在表現が得られます．

In [ ]:
# 1枚の画像を，学習済みCNNの最初の畳み込み層へ通す
sample_image, sample_label = test_dataset[0]

sample_batch = (
    sample_image
    .unsqueeze(0)
    .to(device)
)

model.eval()

with torch.no_grad():
    first_conv_output = model.features[0](
        sample_batch
    )

    first_feature_maps = model.features[1](
        first_conv_output
    )

first_feature_maps = (
    first_feature_maps
    .squeeze(0)
    .cpu()
)

# 最初の8枚だけ可視化する
fig, axes = plt.subplots(
    2,
    4,
    figsize=(10, 5),
)

for filter_index, ax in enumerate(axes.flat):
    ax.imshow(
        first_feature_maps[
            filter_index
        ],
        cmap="gray",
    )

    ax.set_title(
        f"Feature map {filter_index}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

### 1.8. Vision Transformer

画像処理のモデルとして，CNNだけでなく**Vision Transformer**(ViT)も使われています．ここでは，より高度な処理のできるモデルとして紹介します．

Transformerは，自然言語処理や翻訳を通して発展したモデルですが，画像も小さなパッチへ分割して数値的に扱えば，文章中のトークンに近い形で扱うことができます．ViTの考え方は次の5段階で捉えられます．

1. 画像を一定サイズのパッチへ分割する．  
2. 各パッチを埋め込み(ベクトル)へ変換する．  
3. パッチの並びをTransformerへ入力する．  
4. パッチ同士の関係を利用して画像全体の潜在表現を作る．  
5. 最後に分類などのタスクを行う．

今回はViTそのものを学習せず，すでに学習されたモデルを読み込んで，パッチへ分割するところだけ可視化します．

CNNとViTは，画像を扱う方法が異なります．どちらも画像認識に使われますが，画像をどのような単位で表現するかという考え方が異なります．
- CNN: 小さなフィルタを画像上で動かしながら局所的な特徴を捉えます．
- ViT: 画像をパッチの列として扱い，Transformerでパッチ同士の関係を処理します．

In [ ]:
# 28×28画像を7×7のパッチへ分ける
image, label = test_dataset[0]

patch_size = 7
image_2d = image.squeeze()
patches = []

for row in range(0, 28, patch_size):
    for col in range(0, 28, patch_size):
        patches.append(
            image_2d[
                row:row + patch_size,
                col:col + patch_size,
            ]
        )

fig, axes = plt.subplots(
    4,
    4,
    figsize=(7, 7),
)

for index, ax in enumerate(axes.flat):
    ax.imshow(
        patches[index],
        cmap="gray",
    )
    ax.set_title(f"Patch {index}")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("パッチ数:", len(patches))

## 2. 自然言語処理

画像処理では，画像を画素値の集まりとしてモデルへ入力しました．文章でも考え方は同じで，人間にとって文章は文字や単語の並びですが，ニューラルネットワークは文字列そのものを直接計算できません．

そこで，自然言語処理では大まかに次の処理を行います．

1. 文章をトークンと呼ばれる単位へ分割する．
2. トークンを**整数のIDへ変換**し，さらに**埋め込み**(ベクトル)へ変換してからモデルに入力する．
3. Transformerなどのモデルでトークン同士の関係を処理する．
4. 作られた潜在表現を利用して，分類，生成，要約などのタスクを行う．

今回は，まず英文の感情分類を実際に動かし，そのあとに「Transformerの中では何をしているのか」を少しずつ確認します．

### 2.1. まず事前学習済みモデルを動かしてみる

今回は，目的のタスクを十分な精度でこなせるよう学習されている「事前学習済みモデル」として，DistilBERTと呼ばれるものを利用します．ここでは，モデルを最初から学習する必要はありません．

そして，入力した英文に対して，それが積極的(Positive)か消極的(Negative)のいずれであるかを出力する分類タスクを行います．

同じ `movie` や `like` という単語が含まれていても，文章全体によって結果が変化します．例えば

- `I did not like this movie.`

という文章では，`like` だけを見ればPositiveかもしれませんが `not` との関係を考える必要があります．

このように，文章を扱うには**単語1個だけではなく，周囲に並ぶトークンとの関係を考えること**が重要です．

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

SENTIMENT_MODEL_NAME = (
    "distilbert/"
    "distilbert-base-uncased-finetuned-sst-2-english"
)

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=SENTIMENT_MODEL_NAME,
    device=(
        0 if torch.cuda.is_available()
        else -1
    ),
)

example_texts = [
    "This movie was really interesting!",
    "This movie was boring.",
    "I did not like this movie.",
    "This movie was not bad.",
]

results = sentiment_analyzer(
    example_texts
)

for text, result in zip(
    example_texts,
    results,
):
    print("入力:", text)
    print("出力:", result)
    print()

### 2.2. トークナイズ

Transformerへ文章を入力する前に，文章をトークンと呼ばれる処理単位へ分割します．このような作業をトークナイズといいます．トークンは必ずしも「英単語1個」と一致するとは限りません．

今回は紹介しませんが，モデルによっては，単語より細かいサブワード単位へ分割されることもあります．また，未知の長い単語が現れても，すでに知っている小さな部分へ分割できれば扱いやすくなります．

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    SENTIMENT_MODEL_NAME
)

tokenization_examples = [
    "I did not like this movie.",
    "This movie was unbelievable.",
    "Machine learning is interesting.",
]

for text in tokenization_examples:
    tokens = tokenizer.tokenize(
        text
    )

    print("入力:", text)
    print("トークン:", tokens)
    print()

### 2.3. トークンIDへの変換

文章をトークンへ分割しただけでは「数値」の形をしていないので，まだニューラルネットワークへ入力できないことです．

そのため，まずは各トークンを**トークンID**と呼ばれる整数へ変換します．

トークンIDは，単語の意味そのものを表す数値ではありません．例えばID `100` のトークンが，ID `50` のトークンの「2倍の意味」を持つわけではなく，モデルが持っている語彙表の中で何番目に登録されているかを表しているにすぎません．

したがって，次にトークンIDに基づいて，ニューラルネットワークの入力にできるような数値表現へ更に変換するプロセスが必要になります．

In [ ]:
text = "I did not like this movie."

encoded = tokenizer(
    text,
    return_tensors="pt",
)

print("入力文:")
print(text)

print("\ninput_ids:")
print(encoded["input_ids"])

print("\nトークンとトークン IDの対応:")
tokens = tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][0]
)

for token, token_id in zip(
    tokens,
    encoded["input_ids"][0].tolist(),
):
    print(
        f"{token:>12s} : {token_id}"
    )

### 2.4. 埋め込み

各トークンに対応するIDから変換されて得られ，ニューラルネットワークの入力とすることが可能な高次元のベクトルを**埋め込み**(embedding)といいます．

特に自然言語処理を行うネットワークでは，このようなIDから変換された埋め込みを入力として，内部で数値の変換を繰り返すことによって，最終的に目的のタスクを解けるようにします．

そのためのネットワークの構成として，第2回で取り上げたような全結合ニューラルネットワーク，画像処理において導入したCNNだけでなく，次に紹介するTransformerなどがあります．

### 2.5. Transformer

ChatGPTの "T" は "Transformer" の略です．Transformerとは，文章を表すトークンの系列同士の関係性をAttention機構というもので捉えながら，目的のタスクを解くための潜在表現を構築するニューラルネットワークのことです．

Transformerでは，文章を処理するとき，あるトークンだけを見るのではなく，文章中にある他のトークンの情報も利用します．例えば

- `The movie was not very good.`

という文章中の単語 `good` を処理するとき，`not` や `very` などの情報を利用できれば，単純に `good` があるからPositiveと判断するよりも文脈を反映した表現を作れます．

### 2.6. Attention

Transformerの内部にあるAttention機構の詳細は説明しませんが，大まかにいうと，入力された系列のうちで重要な部分を捉えて処理するようにしています．

Transformer内部で使われる代表的なAttentionとして，1つの文章の中にあるトークン同士で情報をやり取りする**Self-Attention**があります．例えば

- `The animal did not cross the street because it was tired.`

という文章で，途中の `it` が何を指しているかを考えるには，他のトークンとの関係を見る必要がありますが，Self-Attentionによって，モデルは離れた位置にあるトークンの情報も利用できます．

ただし，ここで大切なのは，Attentionを「人間と同じように意味を理解している仕組み」とはいえません．Transformerは，Attention機構に基づいて，**どのトークン同士の関係を利用すると目的のタスクを上手く解けるか**を学習を通して調整しています．

#### 補足: Query, Key, Value

Transformerを勉強すると，Attentionの説明でQuery, Key, Valueという言葉がよく登場します．

今回は計算式までは扱いませんが，イメージとして次のように考えることができます．

| 名前 | 初学者向けのイメージ |
|---|---|
| Query | 今のトークンは，どの情報を探しているか |
| Key | 各トークンは，どのような情報を持っているかを照合するための手がかり |
| Value | 実際に他のトークンから受け取る情報 |

QueryとKeyを使って「どこを重視するか」を決め，Valueから情報を集めます．この計算を大量のTokenについて並列に行えることが，Transformerの特徴の1つです．

### 2.7. 位置エンコーディング

文章では，単語の順番が重要です．例えば，以下の2つの文を考えます．

- `A dog bites the man.`
- `The man bites a dog.`

これらの文では，使われている単語は同じでも意味が異なります．しかし，純粋なSelf-Attentionにはでは「何番目のトークンか」という順序情報が自動的に十分表現される仕組みはありません．

そこで，Transformerでは，トークンからの埋め込みに位置情報を加える必要があり，そのような手法を**位置エンコーディング**(positional encoding; PE)といいます．実際の実装方法には複数あり，ルールベースで規定する方法もありますし，それさえも学習対象にするという方法もあります．ただし，初学者の段階では，トークンの意味を表す数値だけでなく，文章中の位置を表す情報もモデルへ与えると理解すれば十分です．

### 2.8. Transformerブロック

TransformerはAttention機構だけでできているわけではなく，ブロックとして他の処理もまとめられています．代表的なTransformerブロックには，主に次の処理が含まれます．

1. **Self-Attention**  
   トークン同士の関係を利用して情報を集めます．

2. **Feed Forward Network**(FFN)  
   各トークンの表現を，ニューラルネットワークでさらに変換します．これは，第2回で行った全結合ニューラルネットワークの処理と似ています．

3. **残差結合**(Residual Connection)  
   変換前の情報も残しながら処理します．深いネットワークを学習しやすくするために重要です．

4. **正規化**(Normalization)  
   数値のスケールを整え，学習を安定させます．

このようなブロックを複数重ねることで，浅い層から深い層へ進むにつれて，トークンの潜在表現がうまく変化することを期待します．

### 2.9. Transformerの入力と出力を実際に見る

ここまでは `pipeline` を使って簡単に分類しました．次は，トークナイザとモデルを別々に使い，内部でどのようなTensorが受け渡されているかを確認します．

In [ ]:
text_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        SENTIMENT_MODEL_NAME
    )
    .to(device)
)

text = "This movie was not bad."

inputs = tokenizer(
    text,
    return_tensors="pt",
)

inputs = {
    key: value.to(device)
    for key, value
    in inputs.items()
}

text_model.eval()

with torch.no_grad():
    outputs = text_model(
        **inputs,
        output_hidden_states=True,
    )

print("input_idsの形:")
print(inputs["input_ids"].shape)

print("\nlogitsの形:")
print(outputs.logits.shape)

print("\n最後の層の潜在表現の形:")
print(
    outputs
    .hidden_states[-1]
    .shape
)

例えば `[1，8，768]` のような形が表示された場合，それぞれ次の意味です．

1. `1`: 一度に入力した文章数．
2. `8`: トークン数．
3. `768`: 各トークンを表す潜在表現の次元数．

つまり，各トークンはTransformerを通過したあと，768個の数値からなる埋め込み(ベクトル)として表現されています．このような潜在表現を利用して，最終的に感情分類を行います．ここでも，第2回までと同じ考え方が使われています．

モデルは最初から `POSITIVE` や `NEGATIVE` という文字列を直接出しているわけではなく，まずは各クラスに対応するlogitを出力し，そこから確率や予測クラスを求めています．この点は，Fashion-MNISTの10クラス分類とほぼ同じです．

In [ ]:
probabilities = F.softmax(
    outputs.logits,
    dim=1,
)

predicted_id = probabilities.argmax(
    dim=1
).item()

label_name = text_model.config.id2label[
    predicted_id
]

print("入力文:")
print(text)

print("\nlogits:")
print(
    outputs.logits
    .cpu()
    .numpy()
)

print("\n確率:")
print(
    probabilities
    .cpu()
    .numpy()
)

print("\n予測:")
print(label_name)

### 2.10. BERT系とGPT系は何が違うのか

Transformerと聞くと，BERTやGPTなど様々なモデル名が登場します．細かい違いは多数ありますが，この段階では次の違いを押さえておけば十分です．

| モデルの考え方 | 得意な使い方の例 |
|---|---|
| Encoder型 | 入力された文章全体を読み，分類や情報抽出に使う |
| Decoder型 | これまでのトークンを使って，次のトークンを順番に生成する |
| Encoder-Decoder型 | 入力文章を読み，別の文章を生成する |

今回使っているDistilBERTはEncoder型であり，文章全体を読んで得られる潜在表現から感情分類します．一方，GPT系の大規模言語モデルは主にDecoder型で，次のトークンを予測する処理を繰り返すことで文章を生成します．

いずれもTransformerを基盤としていますが，何を学習させ，どのように使うかが異なります．

### 2.11. 事前学習とファインチューニング

今回のDistilBERTは，この講義中にゼロから学習しているわけではありません．

現在の自然言語処理では，大規模なモデルを毎回ゼロから学習するよりも，まず大量のデータで**事前学習**し，その後で目的に合わせて利用する方法が一般的です．大まかには次の3段階で考えられます．

1. **事前学習**  
   大量の文章から，言語のパターンを学習します．

2. **ファインチューニング**  
   感情分類など，特定のタスク用データを使って追加の学習を行います．

3. **推論**  
   学習済みモデルへ新しい文章を入力し，分類や生成を行います．

今回利用したモデルは，事前学習済みDistilBERTをSST-2という感情分類タスク向けにファインチューニングしたものです．

### 2.12. Transformerは何でも理解できるのか

Transformerは非常に高性能ですが，文章を人間と全く同じように理解していると考えるのは適切ではありません．例えば，次のような入力では難しくなる場合があります．

- 皮肉を含む文章．
- 非常に長い前後関係が必要な文章．
- 学習データにほとんど存在しなかった専門用語．
- 曖昧な表現．
- 事実確認が必要な文章．

モデルの出力を見るときは，精度だけでなく，「どのようなデータで学習され，何を目的として最適化されたモデルなのか」を考えることが重要です．

### 2.13. 自分で文章を変えてみる

最後に，モデルを眺めるだけでなく，入力を変えて動かしてみてください．

In [ ]:
my_texts = [
    "I really enjoyed this class.",
    "I did not enjoy this class.",
    "This class was not bad.",
    "The lecture was difficult, but very interesting.",
]

my_results = sentiment_analyzer(
    my_texts
)

for text, result in zip(
    my_texts,
    my_results,
):
    print("入力:", text)
    print("結果:", result)
    print()

## 3. 画像と言語の両方を扱うモデル

ここまでは，画像処理と言語処理を別々に扱いました．しかし，現在のモデルでは，以下のように異種の情報を扱うことが多いです．
- 画像について質問する．
- 文章から画像を検索する．
- 画像に説明文を付ける．
- 画像と文章が一致しているか判定する．
- 画像を見ながら文章を生成する．

画像や音声，文章など，情報を表現する形態を**モダリティ**といい，複数のモダリティを組み合わせるモデルを**マルチモーダルモデル**といいます．

### 3.1. 画像と文章は，そのままでは比較できない

本来，画像は画素値の集まり，文章はトークンの並びとして表現されており，その状態では「ある画像」と「ある文章」がどの程度似ているかという関係を直接比較できません．

そこで，画像と文章をそれぞれニューラルネットワークへ入力し，**同じ次元数の潜在表現**へ変換することにより，潜在空間において画像と文章の近さを比較できます．

### 3.2. CLIP

CLIPは，画像と文章を対応付けて扱う代表的なモデルであり，大きく3つの部分で考えられます．

1. **画像エンコーダ**  
   入力された画像を潜在表現へ変換します．

2. **テキストエンコーダ**  
   入力された文章を潜在表現へ変換します．

3. **類似度の計算**  
   画像からの潜在表現と，文章からの潜在表現との類似度を比較します．

今回使う事前学習済みモデル `openai/clip-vit-base-patch32` では，画像側にVision Transformer，文章側にTransformer系のテキストエンコーダが使われています．

つまり，第1章で少し扱ったVision Transformerと，第2章で扱ったTransformerが1つのモデルの中でつながります．

### 3.3. CLIPはどのように学習されるのか

CLIPでは，画像とその画像を説明する文章のペアを大量に用いて学習します．例えば，以下のような組み合わせがあるとします．

| 画像 | 対応する文章 |
|---|---|
| 犬の画像 | `a photo of a dog` |
| 自動車の画像 | `a photo of a car` |
| 海の画像 | `a photo of the ocean` |

このようなデータに基づく学習では，元々ペアとなっている「画像と文章」のそれぞれに対応する潜在表現が，潜在空間内において近くなるようにします．同時に，ペアとなっていない「画像と文章」のそれぞれに対応する潜在表現は，潜在空間内において離れるようにします．このような学習方法を**対照学習**(contrastive learning)といいます．

このように，対照学習で「**正しい組み合わせを近づけ，間違った組み合わせを離す**」ための損失関数としてはInfoNCE損失などが用いられますが，その詳細はここでは説明しません．

### 3.4. Zero-shot分類

通常の画像分類モデルでは，例えばFashion-MNISTなら `T-shirt/top`, `Trouser`, `Pullover` などの10クラスをあらかじめ決めて学習しました．

CLIPでは，候補となるクラス名を文章として入力できます．

例えば，

- `a photo of a shirt`
- `a photo of sneakers`
- `a photo of a bag`

という文章をモデルへ入れ，画像との類似度を比較できます．

このように，特定の分類器をそのクラス専用に追加学習していなくても，文章でクラス候補を与えて分類し，モデルの性能を試したり実際に使用することを**Zero-shot分類**と呼びます．

### 3.5. CLIPを実際に動かす

本体のCLIPは，自然画像と文章の大規模なペアで学習されています．Fashion-MNISTは28×28画素の小さなグレースケール画像なので，CLIPにとって得意な条件ではありません．そのため，ここでは高い精度を狙うのではなく，**画像と文章を同じモデルへ入力し，対応度を比較できること**を見ます．

最後に表示される `similarity` は，今回与えた文章候補の中で，どの文章が画像と対応しているとモデルが判断したかを比較する値です．重要なのは，通常の10クラス分類器とは違い，**候補となるクラスを文章として変更できる**ことです．

In [ ]:
fashion_transform = transforms.ToTensor()

fashion_test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=fashion_transform,
)

fashion_class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

sample_index = 0
sample_image, true_label = fashion_test_dataset[
    sample_index
]

plt.figure(figsize=(4, 4))

plt.imshow(
    sample_image.squeeze(),
    cmap="gray",
)

plt.title(
    fashion_class_names[
        true_label
    ]
)

plt.axis("off")
plt.show()

In [ ]:
CLIP_MODEL_NAME = (
    "openai/clip-vit-base-patch32"
)

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL_NAME
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_MODEL_NAME
)

clip_model.eval()

In [ ]:
pil_image = transforms.ToPILImage()(
    sample_image
).convert("RGB")

text_candidates = [
    "a photo of a t-shirt",
    "a photo of trousers",
    "a photo of a pullover",
    "a photo of a dress",
    "a photo of a coat",
    "a photo of sandals",
    "a photo of a shirt",
    "a photo of sneakers",
    "a photo of a bag",
    "a photo of ankle boots",
]

clip_inputs = clip_processor(
    text=text_candidates,
    images=pil_image,
    return_tensors="pt",
    padding=True,
)

clip_inputs = {
    key: value.to(device)
    for key, value
    in clip_inputs.items()
}

with torch.no_grad():
    clip_outputs = clip_model(
        **clip_inputs
    )

similarities = (
    clip_outputs
    .logits_per_image
    .softmax(dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

result_df = pd.DataFrame({
    "text": text_candidates,
    "similarity": similarities,
}).sort_values(
    "similarity",
    ascending=False,
)

display(result_df)

### 3.6. 潜在表現を確認する

CLIPの中では，画像と文章が最終的に同じ次元数の潜在表現へ変換されます．実際に形を確認してみます．

画像は1枚なので画像由来の潜在表現は1個，文章候補は10個なので文章由来の潜在表現は10個あります．どちらも同じ次元数の潜在表現になっているため，画像と文章の近さを計算できます．

この「**異なる種類のデータを共通の表現空間へ変換する**」という考え方は，マルチモーダルモデルの学習で非常に重要です．

In [ ]:
with torch.no_grad():
    image_features = clip_outputs.image_embeds
    text_features = clip_outputs.text_embeds

print(
    "画像由来の潜在表現の形:",
    image_features.shape,
)

print(
    "文章由来の潜在表現の形:",
    text_features.shape,
)

### 3.7. CLIPと生成AIは同じものではない

CLIPは，画像と文章を対応付けることが得意ですが，それ自体はChatGPTのように長い文章を生成するモデルではありません．

画像について文章で回答したり，画像を見ながら会話したりするモデルでは，CLIPとは別の構成が使われます．現在の画像と言語を扱う生成モデルでは，典型的には次のような構成が使われます．

1. **Vision Encoder**  
   Vision Transformerなどを使って画像を潜在表現へ変換します．

2. **ProjectorやAdapter**  
   画像の潜在表現を，言語モデルが扱いやすい形式へ変換します．

3. **視覚言語モデル**(Vision Language Model; VLM)  
   画像から得た情報と文章のトークンを利用して，次のトークンを生成します．

実際のモデルによって構成は異なりますが，このように画像処理と自然言語処理を接続することで，画像を見ながら文章を生成できるようになります．

### 3.8. マルチモーダルモデルでは何が難しいのか

画像と文章を同時に扱えるようになるとできることは増えますが，問題も増えます．例えば次のような問題が生じることがあります．

- 画像の細かい位置関係を正確に読み取れない．
- 画像に存在しない物体について説明してしまう．
- 文字が小さい画像や複雑な図表を苦手とする．
- 文章と画像のどちらを重視するかによって結果が変わる．
- 学習データに含まれる偏りを引き継ぐ．

したがって，画像と言語を扱えるからといって，「画像を人間と同じように完全に理解している」とは限りません．

## 4. まとめ: 画像処理，言語処理，マルチモーダルモデル

ここまで登場したモデルを整理します．

| モデル | 主な入力 | 今回の役割 | 主なポイント |
|---|---|---|---|
| CNN | 画像 | Fashion-MNIST分類 | 局所的な特徴を畳み込みで捉える |
| Vision Transformer | 画像パッチ | 画像をTransformerで扱う | パッチ同士の関係を処理する |
| DistilBERT | トークン化した文章 | 感情分類 | トークン同士の関係から潜在表現を作る |
| CLIP | 画像と文章 | 画像と文章の対応度比較 | 異なるデータを共通の潜在空間へ変換する |
| 生成型VLM | 画像と文章 | 画像について文章を生成する | Vision Encoderと言語モデルを接続する |

モデル名は増えましたが，初学者の段階で「名前を全部暗記する」必要はありません．それぞれについて，以下の観点で整理すると良いと思います．

1. 何を入力するのか．
2. モデルの中でどのような潜在表現へ変換するのか．
3. 何を出力するのか．
4. 学習する場合，どのような正解やLossを使うのか．
5. 結局，どういう個性があるのか．

### 4.1. モデル構築の流れは共通している

第1回から繰り返し扱ってきたモデル構築の流れに戻します．入力データやモデルが複雑になっても，この大きな流れは変わりません．

1. データを準備する  
    画像ならTensor化，文章ならトークナイズ，画像と文章を扱うなら両方の前処理を行います．

2. モデルを準備する  
    問題に応じてCNN，Transformer，CLIPなどを選びます．

3. (必要に応じて)学習する  
    ゼロから学習する場合もあれば，事前学習済みモデルをファインチューニングする場合もあります．学習する場合は，これまでと同じように予測と正解から損失を計算し，誤差逆伝播法でパラメータを更新します．

4. テスト・推論する  
    未知データを入力し，分類結果，生成文章，画像と文章の類似度などを確認します．

### 4.2. 今回の到達点

今回の内容で最も重要なのは，CNNやTransformerの細かい数式を覚えることではありません．

次の内容がわかっていれば大変良いと思います．

1. 画像は画素，文章はトークンとして数値化してモデルへ入力する．
2. CNNは画像の局所的な特徴を学習できる．
3. Vision Transformerでは，画像をパッチへ分割したうえでTransformerに入力している．
4. Transformerはトークン同士の関係を利用しながら潜在表現を作る．
5. 事前学習済みモデルを利用すれば，大規模モデルを毎回ゼロから学習する必要はない．
6. CLIPでは画像と文章を共通の潜在空間へ変換して比較する．
7. 画像と言語を扱う生成モデルでは，Vision Encoderと言語モデルを接続する考え方が使われる．
8. どのモデルでも，「データ準備，モデル準備，学習，テスト・推論」という基本的な見方ができる．

ここまで理解できれば，今後CNN，ViT，BERT，GPT，CLIP，VLMなど新しいモデル名が登場しても，「何を入力し，どのような表現へ変換し，何を出力するのか」という観点から整理できます．